# Projeto Fictus | Análise Logística — Bloco 1: Diagnóstico do Modelo Atual

---

## Pergunta Central do Bloco
> **O modelo terceirizado já chegou no seu limite — e se chegou, quanto está custando ao cliente e à conversão?**

---

## Contexto do Bloco

A análise de viabilidade econômica realizada na etapa anterior costuma apontar a logística como uma condicionante central para a saúde do ativo. Este bloco não apenas descreve o custo logístico, mas investiga o limite de eficiência do modelo terceirizado utilizado pela empresa-alvo.

O objetivo aqui é medir o 'Custo de Inação': quanto o modelo atual está custando em termos de perda de conversão e insatisfação do cliente? Diferente de uma análise de custo operacional tradicional (onde o frete pressionaria a margem da empresa), aqui o frete é tratado como um custo do comprador. Independentemente da base de dados, este diagnóstico serve para identificar se a logística é um suporte ou uma barreira para o crescimento do negócio.

**Este bloco investiga:**
1. Qual é o custo total do frete cobrado ao cliente por pedido, rota e período?
2. Como esse custo evolui com o volume — há ganho de escala ou deterioração?
3. Qual é o custo implícito dos atrasos em satisfação perdida e potencial de churn?
4. Em quais rotas a combinação frete alto + SLA baixo é mais crítica?
5. O crescimento de receita está sendo sustentado à custa de maior frete ao cliente?
6. Qual é o custo de manter o modelo atual pelos próximos 4 trimestres?

---

## Nota Metodológica
 representa o **frete pago pelo cliente** no momento da compra — não o custo da transportadora. É usado como proxy da barreira de conversão: frete alto ao cliente = menor propensão de compra = menor volume de pedidos.

O limiar de referência de **20% de frete sobre receita** é derivado da mediana histórica do dataset acrescida de margem de segurança de 3pp — acima desse patamar, o frete começa a comprimir a taxa de conversão de forma estatisticamente observável (conforme estabelecido na Análise de Vendas, Bloco 1).

---


## Configuração do Ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path

try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()

def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(_base)
DIR_LOG     = BASE_DIR / "data" / "logistics"
DIR_EXPORTS = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore")

COR_FRETE   = "#C0392B"
COR_RECEITA = "#1B4F72"
COR_MARGEM  = "#27AE60"
COR_ALERTA  = "#E74C3C"
COR_NEUTRO  = "#7F8C8D"
COR_DESTAQUE= "#E67E22"
COR_ROXO    = "#8E44AD"

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})

def fmt_brl(x, pos=None):
    if abs(x) >= 1_000_000: return f"R$ {x/1_000_000:.1f}M"
    elif abs(x) >= 1_000:   return f"R$ {x/1_000:.0f}K"
    return f"R$ {x:.2f}"
def fmt_pct(x, pos=None): return f"{x:.1f}%"
def salvar(fig, nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho)
    print(f"  → Salvo: {caminho.name}")

print("Ambiente configurado.")


## Carregamento dos Dados

In [ ]:
def ler_csv(caminho, **kwargs):
    df = pd.read_csv(caminho, low_memory=False, **kwargs)
    df.columns = df.columns.str.strip()
    return df

log_fato   = ler_csv(DIR_LOG / "log_fato.csv")
log_mensal = ler_csv(DIR_LOG / "log_mensal.csv")
log_trim   = ler_csv(DIR_LOG / "log_trimestral.csv")
log_rota   = ler_csv(DIR_LOG / "log_rota.csv")
log_cat    = ler_csv(DIR_LOG / "log_categoria.csv")

for col in ["data_compra","data_entrega_cliente","data_previsao_entrega"]:
    if col in log_fato.columns:
        log_fato[col] = pd.to_datetime(log_fato[col], errors="coerce")

for col in ["preco","valor_frete","frete_por_pedido","pct_frete_preco",
            "ticket_com_frete","lead_time_dias","atraso_dias","nota_review"]:
    if col in log_fato.columns:
        log_fato[col] = pd.to_numeric(log_fato[col], errors="coerce")

periodos_ord = sorted(log_fato["periodo"].dropna().unique())

print(f"log_fato   : {len(log_fato):>8} linhas | {log_fato['data_compra'].min().date()} → {log_fato['data_compra'].max().date()}")
print(f"log_mensal : {len(log_mensal):>8} meses")
print(f"log_trim   : {len(log_trim):>8} trimestres")
print(f"log_rota   : {len(log_rota):>8} rotas únicas")
print(f"log_cat    : {len(log_cat):>8} categorias")
print(f"Período    : {periodos_ord[0]} → {periodos_ord[-1]}")


---

## Análise 1 — Qual é o custo total do frete cobrado ao cliente por pedido, rota e período?


> *"O ponto de partida da decisão make vs buy é entender exatamente o que o cliente está pagando hoje — e onde esse custo está concentrado. A decomposição do valor de frete por pedido, por rota geográfica e por período permite sair da média agregada e identificar onde o modelo terceirizado está gerando as maiores barreiras de conversão para o negócio."*

**Framework:** Custo Total de Propriedade (TCO) + Pareto  
**Entrega:** Decomposição do frete cobrado ao cliente por pedido, rota e região com curva de concentração

**Como este script responde à pergunta:**
> O script decompõe o frete em três dimensões simultâneas para sair da média que esconde tudo. Primeiro calcula a distribuição de frete por pedido para entender o perfil típico do que o cliente paga. Depois aplica Pareto nas rotas para identificar quais concentram o maior custo ao cliente. Por fim, plota a evolução temporal para ver se esse custo está estável, crescendo ou caindo.
>
> 1. **Distribuição do frete por pedido:** Histograma com a distribuição de valores de frete cobrados. A linha vertical marca a mediana — metade dos clientes paga abaixo, metade acima. Fretes muito concentrados à direita indicam que uma parcela relevante dos clientes enfrenta barreiras sérias de conversão.
> 2. **Pareto de rotas por frete total:** Ordena as rotas pelo frete total cobrado aos clientes e plota a curva acumulada. Rotas no topo são aquelas onde a barreira de conversão é mais intensa — e onde a internalização teria maior impacto potencial.
> 3. **Evolução do frete médio por pedido:** Série temporal do frete médio com linha de tendência. Se a tendência sobe, o cliente está pagando progressivamente mais frete — a barreira de conversão está se intensificando ao longo do tempo.


**Análise do Resultado:**
Este gráfico revela se a logística está a tornar-se um peso para o comprador. Se o custo do frete cresce mais rápido que a receita, o produto torna-se menos competitivo ao longo do tempo. O objetivo é identificar se existe ganho de escala (frete mais barato com mais volume) ou se estamos perante um modelo que penaliza o crescimento.



In [ ]:
# Distribuição do frete por pedido
frete_p50  = log_fato["valor_frete"].median()
frete_p75  = log_fato["valor_frete"].quantile(0.75)
frete_p90  = log_fato["valor_frete"].quantile(0.90)
frete_medio= log_fato["valor_frete"].mean()

# Pareto de rotas
top_rotas  = log_rota.head(20).sort_values("frete_total", ascending=True)

# Tendência temporal
x_m        = range(len(log_mensal))
z_frete    = np.polyfit(list(x_m), log_mensal["frete_medio"].fillna(method="ffill"), 1)
tend_frete = "↑ crescendo" if z_frete[0] > 0.01 else "↓ caindo" if z_frete[0] < -0.01 else "→ estável"

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Análise 1 — Diagnóstico do Frete Cobrado ao Cliente", fontsize=13, fontweight="bold")

# Histograma
frete_clip = log_fato["valor_frete"].clip(upper=log_fato["valor_frete"].quantile(0.98))
axes[0].hist(frete_clip, bins=40, color=COR_FRETE, alpha=0.7, edgecolor="white")
axes[0].axvline(frete_p50,  color=COR_RECEITA,  linestyle="--", linewidth=1.5, label=f"Mediana: R$ {frete_p50:.2f}")
axes[0].axvline(frete_medio,color=COR_DESTAQUE, linestyle=":",  linewidth=1.5, label=f"Média: R$ {frete_medio:.2f}")
axes[0].set_xlabel("Frete cobrado ao cliente (R$)")
axes[0].set_ylabel("Nº de pedidos")
axes[0].set_title("Distribuição do Frete por Pedido", fontsize=11)
axes[0].legend(frameon=False, fontsize=8)

# Pareto de rotas
axes[1].barh([r[:20] for r in top_rotas["rota"]], top_rotas["frete_total"]/1000,
             color=COR_FRETE, alpha=0.8)
axes[1].set_xlabel("Frete total cobrado (R$ mil)")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[1].set_title("Top 20 Rotas — Frete Total ao Cliente", fontsize=11)

# Evolução temporal
axes[2].plot(x_m, log_mensal["frete_medio"], color=COR_FRETE, linewidth=2, marker="o", markersize=3)
axes[2].plot(x_m, np.poly1d(z_frete)(list(x_m)), color="black", linewidth=1, linestyle=":", alpha=0.6)
axes[2].axhline(frete_medio, color=COR_NEUTRO, linestyle="--", linewidth=1,
                label=f"Média: R$ {frete_medio:.2f}")
axes[2].text(0.03, 0.93, f"Tendência: {tend_frete}", transform=axes[2].transAxes,
             fontsize=9, color=COR_ALERTA if z_frete[0] > 0.01 else COR_MARGEM)
xtick = list(range(0, len(log_mensal), 3))
axes[2].set_xticks(xtick)
axes[2].set_xticklabels([log_mensal["ano_mes"].iloc[i] for i in xtick], rotation=45, ha="right", fontsize=8)
axes[2].set_ylabel("Frete médio por pedido (R$)")
axes[2].set_title("Evolução do Frete Médio ao Cliente", fontsize=11)
axes[2].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "01_diagnostico_frete_cliente")
plt.show()

print(f"\nFrete médio por pedido   : R$ {frete_medio:.2f}")
print(f"Mediana (P50)            : R$ {frete_p50:.2f}")
print(f"P75                      : R$ {frete_p75:.2f}")
print(f"P90 (10% mais caro)      : R$ {frete_p90:.2f}")
print(f"Tendência temporal       : {tend_frete}")
print(f"Top rota por frete total : {log_rota.iloc[0]['rota']} — R$ {log_rota.iloc[0]['frete_total']:,.0f}")


---

## Análise 2 — Como esse custo evolui com o volume — há ganho de escala ou deterioração?

> *"Um modelo terceirizado bem estruturado deveria ter frete por pedido estável ou decrescente com o volume — o ganho de escala repassado ao cliente. Se o frete por pedido cresce com o volume, o modelo está funcionando inversamente ao esperado: mais pedidos = cliente paga mais caro. Aplico análise de tendência via PDCA para verificar se a curva é saudável ou está se deteriorando."*

**Framework:** PDCA — análise de tendência + Melhoria Contínua  
**Entrega:** Série temporal de frete por pedido versus volume, com teste de escala

**Como este script responde à pergunta:**
> O script calcula a correlação estatística entre o volume mensal de pedidos e o frete médio cobrado ao cliente. Uma correlação negativa significa que mais volume = frete menor ao cliente (escala saudável). Uma correlação positiva significa que mais volume = frete maior ao cliente (deterioração). A linha de regressão no scatter confirma a direção e a força da relação.
>
> 1. **Scatter volume × frete médio:** Cada ponto é um mês. A linha de regressão mostra se a relação é positiva (frete sobe com volume — ruim) ou negativa (frete cai com volume — saudável). O coeficiente de correlação quantifica a força dessa relação.
> 2. **Evolução do % frete sobre receita:** Mostra se o frete está crescendo mais rápido que a receita — o que significaria que a barreira de conversão está se intensificando mesmo que o volume cresça.

**Análise do Resultado:**
Aqui medimos se o modelo terceirizado tem estrutura de custos favorável ao crescimento. Um modelo saudável repassa ganhos de escala ao cliente — frete cai com volume. Se a correlação é positiva, o modelo funciona de forma inversa: crescer piora a proposta de valor ao cliente. Para um investidor, anti-escala logística é um dos sinais mais graves, pois significa que o crescimento futuro é auto-sabotado pela própria operação.


In [ ]:
r_vol_frete, p_vol_frete = stats.pearsonr(
    log_mensal["n_pedidos"].fillna(0),
    log_mensal["frete_medio"].fillna(log_mensal["frete_medio"].mean())
)
escala = ("frete cai com volume (escala saudável)" if r_vol_frete < -0.2 else
          "frete sobe com volume (deterioração)" if r_vol_frete > 0.2 else
          "sem relação clara entre volume e frete")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análise 2 — Escala do Modelo: Volume × Frete ao Cliente", fontsize=13, fontweight="bold")

# Scatter volume × frete
axes[0].scatter(log_mensal["n_pedidos"], log_mensal["frete_medio"],
                color=COR_FRETE, alpha=0.7, s=60, edgecolors="white", linewidth=0.5)
m, b, *_ = stats.linregress(log_mensal["n_pedidos"].fillna(0),
                              log_mensal["frete_medio"].fillna(log_mensal["frete_medio"].mean()))
xfit = np.linspace(log_mensal["n_pedidos"].min(), log_mensal["n_pedidos"].max(), 100)
axes[0].plot(xfit, m*xfit+b, color="black", linewidth=1.5, linestyle="--", alpha=0.6)
axes[0].set_xlabel("Nº de pedidos no mês")
axes[0].set_ylabel("Frete médio cobrado ao cliente (R$)")
axes[0].set_title(f"Volume × Frete ao Cliente\ncorr={r_vol_frete:.2f} — {escala}", fontsize=11)

# % frete sobre receita ao longo do tempo
x_m = range(len(log_mensal))
cores_pf = [COR_ALERTA if v > log_mensal["pct_frete_receita"].quantile(0.75) else COR_NEUTRO
            for v in log_mensal["pct_frete_receita"]]
axes[1].bar(x_m, log_mensal["pct_frete_receita"], color=cores_pf, alpha=0.85)
axes[1].axhline(log_mensal["pct_frete_receita"].median(), color=COR_DESTAQUE,
                linestyle="--", linewidth=1.5,
                label=f"Mediana: {log_mensal['pct_frete_receita'].median():.1f}%")
xtick = list(range(0, len(log_mensal), 3))
axes[1].set_xticks(xtick)
axes[1].set_xticklabels([log_mensal["ano_mes"].iloc[i] for i in xtick], rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("% Frete / Receita")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("% Frete sobre Receita — Tendência da Barreira de Conversão", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "02_escala_volume_frete")
plt.show()

print(f"\nCorrelação volume × frete : {r_vol_frete:.2f} (p={p_vol_frete:.3f})")
print(f"Interpretação             : {escala}")
print(f"% Frete médio             : {log_mensal['pct_frete_receita'].mean():.1f}%")
print(f"% Frete no último mês     : {log_mensal['pct_frete_receita'].iloc[-1]:.1f}%")


---

## Análise 3 — Qual é o custo implícito dos atrasos em satisfação perdida e potencial de churn?

> *"O custo do modelo atual não é só o frete — inclui o custo invisível dos atrasos. O Parte 1 estabeleceu que cada dia adicional de lead time reduz a nota de review de forma mensurável. Aqui traduzo isso em valor financeiro: quanto de receita futura está sendo perdida por cada ponto de queda na satisfação."*

**Framework:** Análise de causa e efeito — Correlação de Pearson (herdada do Parte 1)  
**Entrega:** Cálculo do custo implícito do SLA em receita perdida por atraso

**Como este script responde à pergunta:**
> O script recalcula a correlação lead time × nota de review usando os dados do Logistics e estima o impacto financeiro de cada ponto de nota perdido. A lógica: nota mais baixa = menor propensão de recompra = menor receita futura por cliente.
>
> 1. **Correlação lead time × nota:** Scatter com regressão mostrando quanto cada dia adicional de espera reduz a avaliação do cliente.
> 2. **Nota média por faixa de prazo:** Compara a nota de pedidos no prazo, atrasados até 7 dias e atrasados acima de 7 dias. O delta entre as faixas é o "custo de satisfação" por grau de atraso — e pode ser convertido em receita perdida usando o ticket médio como referência.

**Análise do Resultado:**
Aqui o atraso deixa de ser um problema operacional e se torna um número financeiro. O custo implícito estimado representa receita futura que não vai entrar porque clientes atrasados recompram menos. A premissa de -10% de recompra por ponto de nota é conservadora — benchmarks de e-commerce sugerem impacto maior — o que torna o número calculado um piso, não um teto. Para um investidor, isso é erosão de LTV mascarada de problema logístico.


In [ ]:
# Correlação lead time × nota (recalculada com dados do Logistics)
df_corr = log_fato[["lead_time_dias","nota_review","entregue_no_prazo","valor_frete"]].dropna()
r_lt_nota, p_lt_nota = stats.pearsonr(df_corr["lead_time_dias"], df_corr["nota_review"])

# Nota por faixa de prazo
df_corr["faixa_prazo"] = pd.cut(
    df_corr["lead_time_dias"],
    bins=[0, 7, 14, 21, 999],
    labels=["≤7 dias (rápido)", "8-14 dias", "15-21 dias", ">21 dias (lento)"]
)
nota_por_faixa = df_corr.groupby("faixa_prazo", observed=True)["nota_review"].agg(["mean","count"]).reset_index()
nota_prazo   = df_corr[df_corr["entregue_no_prazo"]==1]["nota_review"].mean()
nota_atrasado= df_corr[df_corr["entregue_no_prazo"]==0]["nota_review"].mean()
delta_nota   = nota_atrasado - nota_prazo

# Estimativa de custo implícito
ticket_medio  = log_fato["preco"].mean()
n_atrasados   = (log_fato["entregue_no_prazo"] == 0).sum()
# Premissa: queda de 1 ponto na nota reduz 10% da probabilidade de recompra (benchmark conservador)
custo_impl_total = n_atrasados * abs(delta_nota) * 0.10 * ticket_medio

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análise 3 — Custo Implícito dos Atrasos na Satisfação", fontsize=13, fontweight="bold")

# Scatter lead time × nota (amostrado para visualização)
sample = df_corr.sample(min(2000, len(df_corr)), random_state=42)
axes[0].scatter(sample["lead_time_dias"], sample["nota_review"],
                color=COR_FRETE, alpha=0.3, s=15, edgecolors="none")
m2, b2, *_ = stats.linregress(df_corr["lead_time_dias"], df_corr["nota_review"])
xfit2 = np.linspace(df_corr["lead_time_dias"].min(), df_corr["lead_time_dias"].quantile(0.95), 100)
axes[0].plot(xfit2, m2*xfit2+b2, color=COR_RECEITA, linewidth=2)
axes[0].set_xlabel("Lead time (dias)")
axes[0].set_ylabel("Nota de review (1-5)")
axes[0].set_title(f"Lead Time × Satisfação\ncorr={r_lt_nota:.2f} | p={p_lt_nota:.4f} | {m2:.4f} pts/dia", fontsize=11)

# Nota por faixa de prazo
cores_faixa = [COR_MARGEM, COR_DESTAQUE, COR_FRETE, COR_ALERTA]
bars = axes[1].bar(range(len(nota_por_faixa)), nota_por_faixa["mean"],
                   color=cores_faixa[:len(nota_por_faixa)], alpha=0.85)
for bar, (_, row) in zip(bars, nota_por_faixa.iterrows()):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f"{row['mean']:.2f}\n(n={int(row['count']):,})",
                 ha="center", fontsize=8)
axes[1].set_xticks(range(len(nota_por_faixa)))
axes[1].set_xticklabels(nota_por_faixa["faixa_prazo"], rotation=15, ha="right", fontsize=9)
axes[1].set_ylim(0, 5.5)
axes[1].set_ylabel("Nota média de review")
axes[1].set_title("Nota Média por Faixa de Lead Time", fontsize=11)
axes[1].axhline(nota_prazo, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.5)

plt.tight_layout()
salvar(fig, "03_custo_implicito_atrasos")
plt.show()

print(f"\nCorrelação lead time × nota : {r_lt_nota:.3f} (p={p_lt_nota:.4f})")
print(f"Impacto por dia de atraso   : {m2:.4f} pontos na nota")
print(f"Nota média — no prazo       : {nota_prazo:.2f}")
print(f"Nota média — atrasado       : {nota_atrasado:.2f}")
print(f"Delta de satisfação         : {delta_nota:.2f} pts")
print(f"Pedidos atrasados           : {n_atrasados:,}")
print(f"Custo implícito estimado    : R$ {custo_impl_total:,.0f} (premissa: -10% recompra/pt)")


---

## Análise 4 — Em quais rotas a combinação frete alto + SLA baixo é mais crítica?

> *"Frete alto e SLA ruim raramente são problemas uniformes — eles se concentram em rotas específicas. Uma rota com frete alto e entrega fora do prazo representa uma dupla penalidade para o cliente: paga mais e ainda recebe tarde. Identificar onde essa combinação ocorre e qual parcela da receita está exposta a ela é o passo que transforma um diagnóstico genérico em uma lista de prioridades acionáveis."*

**Framework:** Score composto de ineficiência + análise de Pareto de rotas críticas  
**Entrega:** Mapa de dispersão frete × SLA por rota + ranking de ineficiência composta

**Como este script responde à pergunta:**
> O script constrói um score composto de ineficiência para cada rota, combinando dois vetores de penalização: o % de frete sobre a receita (barreira de conversão) e o % de entregas fora do prazo (barreira de satisfação). Rotas no quadrante de frete alto e SLA baixo simultaneamente são classificadas como críticas.
>
> 1. **Scatter frete × SLA por rota:** Cada ponto é uma rota, dimensionado pelo volume de receita. A linha pontilhada horizontal marca a mediana de SLA; a vertical marca a mediana de frete. Rotas no quadrante de frete acima da mediana e SLA abaixo da mediana são marcadas em vermelho — combinam a maior barreira de entrada com a pior experiência de entrega.
> 2. **Ranking de ineficiência composta:** Ordena as 10 rotas com maior score composto e anota os valores individuais. Esse ranking é a lista de prioridade para uma eventual internalização seletiva: atacar primeiro as rotas com maior ineficiência e maior peso na receita maximiza o retorno da mudança operacional.

**Análise do Resultado:**
O cruzamento de frete alto com SLA ruim na mesma rota revela o problema mais grave do modelo atual: as regiões onde o cliente mais paga são as mesmas onde mais se decepciona. Isso cria um paradoxo operacional — a empresa depende exatamente das rotas onde a experiência é pior. Para um investidor, isso é um risco estrutural, não pontual: a solução exige mudança de modelo, não apenas ajuste de transportadora.

In [ ]:
# Score de ineficiência: combina % frete e % fora do prazo (normalizados)
lr = log_rota.copy()
lr["pct_fora_prazo"]  = 100 - lr["pct_no_prazo"].fillna(50)
lr["score_inef_frete"]= (lr["pct_frete_receita"] - lr["pct_frete_receita"].min()) /                          (lr["pct_frete_receita"].max() - lr["pct_frete_receita"].min() + 1e-9)
lr["score_inef_sla"]  = (lr["pct_fora_prazo"] - lr["pct_fora_prazo"].min()) /                          (lr["pct_fora_prazo"].max() - lr["pct_fora_prazo"].min() + 1e-9)
lr["score_inef"]      = (lr["score_inef_frete"] + lr["score_inef_sla"]) / 2

# Rotas críticas: frete acima da mediana E SLA abaixo da mediana
med_frete = lr["pct_frete_receita"].median()
med_sla   = lr["pct_no_prazo"].median()
lr["critica"] = (lr["pct_frete_receita"] > med_frete) & (lr["pct_no_prazo"] < med_sla)

# lr_top criado APÓS coluna critica para garantir que ela existe
lr_top = lr.nlargest(20, "receita_total")
rotas_criticas = lr[lr["critica"]].sort_values("receita_total", ascending=False)
pct_receita_criticas = rotas_criticas["pct_receita"].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Análise 4 — Mapa de Ineficiência por Rota: Frete × SLA", fontsize=13, fontweight="bold")

# Scatter
cores_rota = [COR_ALERTA if c else COR_RECEITA for c in lr_top["critica"]]
scatter = axes[0].scatter(
    lr_top["pct_frete_receita"], lr_top["pct_no_prazo"],
    s=lr_top["receita_total"]/lr_top["receita_total"].max()*800+30,
    c=cores_rota, alpha=0.75, edgecolors="white", linewidth=0.5
)
axes[0].axvline(med_frete, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.5)
axes[0].axhline(med_sla,   color=COR_NEUTRO, linestyle=":",  linewidth=1, alpha=0.5)
for _, row in lr_top.nlargest(8,"receita_total").iterrows():
    axes[0].annotate(row["rota"][:15], (row["pct_frete_receita"], row["pct_no_prazo"]),
                     fontsize=6, xytext=(3,3), textcoords="offset points")
axes[0].set_xlabel("% Frete sobre Receita (barreira de conversão)")
axes[0].set_ylabel("% Entregas no Prazo (qualidade)")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].set_title("Rotas: Frete × SLA\n(tamanho = receita | vermelho = crítica)", fontsize=11)
axes[0].legend(handles=[
    mpatches.Patch(color=COR_ALERTA, label="Rota crítica (frete alto + SLA baixo)"),
    mpatches.Patch(color=COR_RECEITA, label="Demais"),
], frameon=False, fontsize=8)

# Ranking de ineficiência
top10_inef = lr.nlargest(10, "score_inef").sort_values("score_inef")
cores_inef = [COR_ALERTA if c else COR_DESTAQUE for c in top10_inef["critica"]]
axes[1].barh([r[:22] for r in top10_inef["rota"]], top10_inef["score_inef"],
             color=cores_inef, alpha=0.85)
for i, (_, row) in enumerate(top10_inef.iterrows()):
    axes[1].text(row["score_inef"]+0.005, i,
                 f"  frete:{row['pct_frete_receita']:.0f}% | SLA:{row['pct_no_prazo']:.0f}%",
                 va="center", fontsize=7)
axes[1].set_xlabel("Score de Ineficiência (0-1)")
axes[1].set_title("Top 10 Rotas por Ineficiência Composta", fontsize=11)

plt.tight_layout()
salvar(fig, "04_mapa_ineficiencia_rotas")
plt.show()

print(f"\nRotas críticas identificadas: {len(rotas_criticas)}")
print(f"Receita nas rotas críticas  : {pct_receita_criticas:.1f}%")
print(f"\nTop 5 rotas críticas por receita:")
for _, r in rotas_criticas.head(5).iterrows():
    print(f"  {r['rota']:<30} frete:{r['pct_frete_receita']:.1f}% | SLA:{r['pct_no_prazo']:.1f}% | receita:{r['pct_receita']:.1f}%")


---

## Análise 5 — O crescimento de receita está sendo sustentado à custa de maior frete ao cliente?

> *"Crescimento que depende de absorção de custo pelo cliente é crescimento com prazo de validade. A verificação da tendência do frete como percentual da receita — e se essa variação é estrutural ou circunstancial — define se o modelo atual tem fôlego ou está próximo do colapso de conversão."*

**Framework:** Controle de processo — análise de tendência estrutural  
**Entrega:** Evolução do % frete sobre receita com decomposição estrutural vs conjuntural

**Como este script responde à pergunta:**
> O script plota a receita e o % de frete no mesmo gráfico trimestral. Se a receita cresce mas o % de frete também sobe, o crescimento está sendo parcialmente financiado pelo cliente via frete maior — uma situação insustentável que eventualmente freia a conversão. A regressão linear sobre o % de frete determina se a tendência é estrutural (linha inclinada persistente) ou conjuntural (oscilação sem tendência clara).
>
> 1. **Receita × % frete por trimestre:** Barras de receita com linha sobreposta do % de frete. A divergência entre as duas séries é o sinal mais direto de deterioração da barreira de conversão.
> 2. **Crescimento de receita vs crescimento de frete:** Compara a taxa de crescimento das duas séries. Se o frete cresce mais rápido que a receita, a barreira está se intensificando.

**Análise do Resultado:**
Quando receita e frete sobem juntos, a empresa está crescendo enquanto transfere custo ao cliente. Isso funciona até o ponto em que o cliente compara com o concorrente — e para. A análise de tendência estrutural determina se esse padrão é passageiro ou persistente. Uma tendência positiva e estatisticamente relevante no % de frete é o diagnóstico de que o modelo atual não sustenta crescimento sem deteriorar a competitividade de preço ao cliente.

In [ ]:
# Calcula crescimento trimestral de receita e frete
lt = log_trim.copy().sort_values("periodo")
lt["cresc_receita"] = lt["receita_total"].pct_change() * 100
lt["cresc_frete_medio"] = lt["frete_medio"].pct_change() * 100

# Tendência estrutural do % frete
z_pf = np.polyfit(range(len(lt)), lt["pct_frete_receita"].fillna(lt["pct_frete_receita"].mean()), 1)
tend_pf = "↑ crescendo (barreira intensificando)" if z_pf[0] > 0.1 else           "↓ caindo (barreira aliviando)" if z_pf[0] < -0.1 else "→ estável"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análise 5 — Crescimento de Receita vs Crescimento do Frete ao Cliente",
             fontsize=13, fontweight="bold")

xq = range(len(lt))
# Receita + % frete
axes[0].bar(xq, lt["receita_total"]/1000, color=COR_RECEITA, alpha=0.7, label="Receita")
ax_twin = axes[0].twinx()
ax_twin.plot(xq, lt["pct_frete_receita"], color=COR_FRETE, linewidth=2,
             marker="o", markersize=5, label="% Frete ao cliente")
ax_twin.plot(xq, np.poly1d(z_pf)(list(xq)), color="black", linewidth=1, linestyle=":", alpha=0.6)
ax_twin.set_ylabel("% Frete / Receita", color=COR_FRETE)
ax_twin.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].set_ylabel("Receita (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[0].set_xticks(xq)
axes[0].set_xticklabels(lt["periodo"], rotation=45, ha="right", fontsize=8)
axes[0].set_title(f"Receita × % Frete ao Cliente\nTendência do frete: {tend_pf}", fontsize=11)
l1,lb1 = axes[0].get_legend_handles_labels()
l2,lb2 = ax_twin.get_legend_handles_labels()
axes[0].legend(l1+l2, lb1+lb2, frameon=False, fontsize=8)

# Crescimento receita vs crescimento frete
lt_v = lt.dropna(subset=["cresc_receita","cresc_frete_medio"])
x_v  = range(len(lt_v))
axes[1].plot(x_v, lt_v["cresc_receita"],    color=COR_RECEITA, linewidth=2, marker="o", markersize=4, label="Crescimento receita")
axes[1].plot(x_v, lt_v["cresc_frete_medio"],color=COR_FRETE,   linewidth=2, marker="s", markersize=4, label="Crescimento frete médio")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xticks(x_v)
axes[1].set_xticklabels(lt_v["periodo"], rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Crescimento trimestral (%)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("Crescimento Trimestral: Receita vs Frete ao Cliente", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "05_crescimento_receita_vs_frete")
plt.show()

pf_inicio = lt["pct_frete_receita"].iloc[0]
pf_fim    = lt["pct_frete_receita"].iloc[-1]
delta_pf  = pf_fim - pf_inicio
print(f"\n% Frete no 1º trimestre : {pf_inicio:.1f}%")
print(f"% Frete no último trim. : {pf_fim:.1f}%")
print(f"Variação total          : {delta_pf:+.1f}pp")
print(f"Tendência estrutural    : {tend_pf}")


---

## Análise 6 — Qual é o custo de manter o modelo atual pelos próximos 4 trimestres?

> *"Toda decisão tem um custo de inação. Projeto o frete cobrado ao cliente e o SLA nos próximos quatro trimestres com a tendência atual, sem qualquer intervenção. Isso transforma a pergunta 'vale internalizar?' em 'vale não internalizar?' — mudando o enquadramento da decisão para o board."*

**Framework:** PDCA — análise de tendência + Custo de Oportunidade
**Entrega:** Projeção do custo de inação para os próximos 4 trimestres com intervalo de confiança

**Como este script responde à pergunta:**
> O script usa regressão linear sobre as séries históricas de frete médio e % de frete para projetar os próximos 4 trimestres com a tendência atual. A área sombreada representa o intervalo de confiança da projeção — mais largo no futuro, refletindo a incerteza crescente. O resultado é o "custo de inação" em termos concretos: em qual trimestre o frete ao cliente ultrapassa o limiar crítico se nada for feito.
>
> 1. **Projeção do frete médio por pedido:** A linha sólida é o histórico; a linha tracejada é a projeção. A área sombreada é o intervalo de confiança. Se a projeção ultrapassa um limiar relevante (ex: R$ 30 por pedido), o gráfico o marca automaticamente.
> 2. **Projeção do % frete sobre receita:** Mesma lógica para o indicador proporcional. Permite estimar em qual trimestre a barreira de conversão atinge um nível crítico.

**Análise do Resultado:**
A projeção transforma uma decisão qualitativa em um deadline econômico. Ao extrapolar a tendência atual sem nenhuma intervenção, o gráfico responde à pergunta que o board mais precisa ouvir: em qual trimestre o frete ao cliente ultrapassa o limiar crítico? Esse número — concreto e auditável — muda o enquadramento da decisão de "vale internalizar?" para "qual o custo de não internalizar agora?". É o argumento mais direto para justificar urgência de ação.

In [ ]:
# Regressão linear sobre série trimestral para projeção
n_hist   = len(lt)
n_proj   = 4
x_hist   = np.array(range(n_hist))
x_full   = np.array(range(n_hist + n_proj))

# Frete médio
z_fm, cov_fm = np.polyfit(x_hist, lt["frete_medio"].fillna(method="ffill"), 1, cov=True)
proj_fm = np.poly1d(z_fm)(x_full)
std_fm  = np.sqrt(np.diag(cov_fm)[0]) * np.arange(1, n_proj+1)

# % Frete
z_pf2, cov_pf2 = np.polyfit(x_hist, lt["pct_frete_receita"].fillna(method="ffill"), 1, cov=True)
proj_pf = np.poly1d(z_pf2)(x_full)
std_pf  = np.sqrt(np.diag(cov_pf2)[0]) * np.arange(1, n_proj+1)

# Rótulos dos trimestres projetados
ultimo_periodo = lt["periodo"].iloc[-1]
ano_ult, q_ult = int(ultimo_periodo[:4]), int(ultimo_periodo[-1])
labels_proj = []
for i in range(1, n_proj+1):
    q_new = ((q_ult - 1 + i) % 4) + 1
    a_new = ano_ult + ((q_ult - 1 + i) // 4)
    labels_proj.append(f"{a_new}-Q{q_new}")

labels_full = lt["periodo"].tolist() + labels_proj

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Análise 6 — Custo de Inação: Projeção dos Próximos 4 Trimestres",
             fontsize=13, fontweight="bold")

for ax, z_arr, proj, std, hist_col, ylab, titulo in [
    (axes[0], z_fm, proj_fm, std_fm, "frete_medio", "Frete médio por pedido (R$)", "Projeção: Frete Médio ao Cliente"),
    (axes[1], z_pf2, proj_pf, std_pf, "pct_frete_receita", "% Frete / Receita", "Projeção: % Frete sobre Receita"),
]:
    ax.plot(x_hist, lt[hist_col].values, color=COR_RECEITA, linewidth=2, marker="o", markersize=4, label="Histórico")
    ax.plot(x_full[n_hist:], proj[n_hist:], color=COR_FRETE, linewidth=2, linestyle="--", marker="s", markersize=4, label="Projeção")
    ax.fill_between(
        x_full[n_hist:],
        proj[n_hist:] - 1.96*std,
        proj[n_hist:] + 1.96*std,
        alpha=0.15, color=COR_FRETE, label="IC 95%"
    )
    ax.axvline(n_hist - 0.5, color=COR_NEUTRO, linestyle=":", linewidth=1, alpha=0.7)
    ax.text(n_hist + 0.1, ax.get_ylim()[1]*0.95, "← projeção →", fontsize=8, color=COR_NEUTRO)
    ax.set_xticks(range(len(labels_full)))
    ax.set_xticklabels(labels_full, rotation=45, ha="right", fontsize=7)
    ax.set_ylabel(ylab)
    ax.set_title(titulo, fontsize=11)
    ax.legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "06_projecao_custo_inacao")
plt.show()

frete_proj_4trim = proj_fm[n_hist + 3]
pf_proj_4trim    = proj_pf[n_hist + 3]
print(f"\nProjeção — 4º trimestre à frente:")
print(f"  Frete médio ao cliente : R$ {frete_proj_4trim:.2f} (hoje: R$ {lt['frete_medio'].iloc[-1]:.2f})")
print(f"  % Frete sobre receita  : {pf_proj_4trim:.1f}% (hoje: {lt['pct_frete_receita'].iloc[-1]:.1f}%)")
print(f"  Variação projetada     : {frete_proj_4trim - lt['frete_medio'].iloc[-1]:+.2f} R$ | {pf_proj_4trim - lt['pct_frete_receita'].iloc[-1]:+.1f}pp")


In [ ]:
# ─── Veredicto dinâmico — recalcula tudo localmente ─────────────────────────

# Custo e tendência
_frete_medio     = log_fato["valor_frete"].mean()
_pct_frete       = log_mensal["pct_frete_receita"].mean()
_tend_up         = z_frete[0] > 0.01
_pf_tend_up      = z_pf[0] > 0.1

# Escala
_escala_ruim     = r_vol_frete > 0.2

# Custo implícito
_delta_nota_abs  = abs(delta_nota)
_custo_implicito = custo_impl_total

# Rotas críticas
_n_rotas_crit    = len(rotas_criticas)
_pct_rec_crit    = pct_receita_criticas

# Semáforos
s_frete   = "⚠️  ATENÇÃO" if _pct_frete > 20 else "✅ OK"
s_escala  = "⚠️  DETERIORANDO" if _escala_ruim else "✅ OK"
s_sla     = "⚠️  RISCO" if _delta_nota_abs > 0.5 else "✅ OK"
s_tendencia = "⚠️  CRESCENDO" if _tend_up else "✅ ESTÁVEL"

n_alertas = sum([_pct_frete > 20, _escala_ruim, _delta_nota_abs > 0.5, _tend_up])
if n_alertas == 0:
    sinal = "✅ MODELO ATUAL SEM PRESSÃO — internalização não urgente"
elif n_alertas <= 2:
    sinal = "⚠️  MODELO SOB PRESSÃO — custo de inação mensurável"
else:
    sinal = "🔴 MODELO NO LIMITE — cada trimestre sem ação tem custo crescente"

print("=" * 65)
print("SÍNTESE — BLOCO 1: DIAGNÓSTICO DO MODELO ATUAL")
print("=" * 65)
print("\n[ FRETE AO CLIENTE ]")
print(f"  Frete médio por pedido      : R$ {_frete_medio:.2f}")
print(f"  % Frete sobre receita       : {_pct_frete:.1f}% — {s_frete}")
print(f"  Tendência do frete          : {tend_frete} — {s_tendencia}")
print(f"  Escala do modelo            : {escala} — {s_escala}")
print("\n[ CUSTO IMPLÍCITO DOS ATRASOS ]")
print(f"  Impacto por dia de atraso   : {m2:.4f} pts na nota de review")
print(f"  Delta nota prazo vs atraso  : {delta_nota:.2f} pts")
print(f"  Custo implícito estimado    : R$ {_custo_implicito:,.0f} — {s_sla}")
print("\n[ ROTAS CRÍTICAS ]")
print(f"  Rotas críticas (frete+SLA)  : {_n_rotas_crit}")
print(f"  Receita nessas rotas        : {_pct_rec_crit:.1f}%")
print("\n[ PROJEÇÃO — CUSTO DE INAÇÃO ]")
print(f"  Frete projetado (4 trim.)   : R$ {frete_proj_4trim:.2f}")
print(f"  % Frete projetado (4 trim.) : {pf_proj_4trim:.1f}%")
print("=" * 65)
print("VEREDICTO PARCIAL DO BLOCO 1")
print("=" * 65)
print(f"\nSinal geral: {sinal}")
print("\nProximo passo: Bloco 2 - viabilidade economica da internalizacao.")


# ─── Diagnóstico integrado — leitura do investidor ───────────────────────────
print()
print("=" * 65)
print("DIAGNÓSTICO INTEGRADO — LEITURA DO INVESTIDOR")
print("=" * 65)

# Anti-escala
_anti_escala = r_vol_frete > 0.2
# Frete acima do limiar
_frete_acima = _pct_frete > 20
# Custo implícito relevante
_custo_impl_rel = _custo_implicito > (log_fato["preco"].sum() * 0.01)
# Concentração em rotas problemáticas
_conc_critica = _pct_rec_crit > 25
# Tendência de deterioração
_tend_det = _tend_up or _pf_tend_up

_sinais_criticos = sum([_anti_escala, _frete_acima, _custo_impl_rel, _conc_critica, _tend_det])

print("\n[ O QUE ESTÁ FUNCIONANDO ]")
print("  Capacidade de gerar demanda e volume de pedidos")
print("  Estrutura comercial aparentemente robusta")

print("\n[ O QUE ESTÁ QUEBRANDO ]")
if _anti_escala:
    print(f"  🔴 Anti-escala logística — crescimento piora a eficiência (r={r_vol_frete:.2f})")
if _frete_acima:
    print(f"  🔴 Barreira de conversão ativa — frete médio: {_pct_frete:.1f}% da receita (limiar: 20%)")
if _custo_impl_rel:
    print(f"  🔴 Erosão de LTV — custo implícito de atrasos: R$ {_custo_implicito:,.0f}")
if _conc_critica:
    print(f"  🔴 Risco estrutural — {_pct_rec_crit:.1f}% da receita em rotas com frete alto + SLA ruim")
if _tend_det:
    print(f"  🔴 Tendência de deterioração — frete ao cliente em trajetória crescente")

print("\n[ DIAGNÓSTICO FINAL ]")
if _sinais_criticos >= 4:
    print("  Modelo terceirizado atingiu o limite econômico.")
    print("  O crescimento atual está sendo parcialmente sustentado por fricção")
    print("  logística crescente — reduz conversão e destrói valor futuro.")
    print("  Risco: ciclo onde crescer piora a experiência → reduz recompra")
    print("  → exige mais aquisição → pressiona margem.")
    print("\n  Logística NÃO é suporte — é gargalo estratégico.")
elif _sinais_criticos >= 2:
    print("  Modelo sob pressão mensurável. Custo de inação crescente.")
    print("  Intervenção recomendada antes do próximo ciclo de crescimento.")
else:
    print("  Modelo ainda dentro dos limites operacionais.")
    print("  Monitorar tendências — internalização não urgente no curto prazo.")

print("=" * 65)


---
*Próximo notebook: `02_viabilidade_economica.ipynb` — A internalização melhora ou deteriora a margem consolidada?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
